## Project 3

by Orsolya Bosáková

### Introduction

The aim of this project is to evaluate the data set of a collection of randomised case-control studies of the effectiveness of descriptive social norms on hotel customers behavior to reuse their towels. The data is obtained from the following study.

Scheibehenne, B., Jamil, T., & Wagenmakers, E.-J. (2016). Bayesian Evidence Synthesis Can Reconcile Seemingly Inconsistent Results: The Case of Hotel Towel Reuse. Psychological Science, 27(7), 1043-1046. https://journals.sagepub.com/doi/10.1177/0956797616644081

We would like to formulate and justify a parametric model for testing the effectiveness of the intervention, estimate the model parameters and test if the intervention has an effect or not.

The data set towelData.csv consist of 7 variables:

    - Source: the identifier code consisting of author name, experiment number and year 

    - AuthorName: the name of the lead author of the study

    - Experiment: the number of the experiment within the respective study

    - Year: the year of the publishing of the study

    - Group: the condition that was assigned to the specific hotel room (Control\Social Norm)

    - Towel.Reuse: the observed response 

    - Count: the total number of guests observed in that particular study and\or cathegory

The following python packages have been used:

In [1]:
import numpy as np
import scipy as sc
import pandas as pd
import bambi as bmb
import arviz as az

g++ not available, if using conda: `conda install gxx`


### Data wrangling

We use the data wrangling technique proposed in the assignment. First, we read the data file and separate the last column of the "count" variable.

In [2]:
# read in data
data_file = "Data/towelData.csv"
data = pd.read_csv(data_file, sep=';', encoding='latin1')
count = data.iloc[:, -1] # get the last column with numbers or yes/no
print(count)

0      74
1      98
2     137
3     124
4     103
5     587
6     174
7     731
8      77
9     406
10     58
11    249
12     82
13    278
14    105
15    277
16     21
17     21
18      4
19      3
20    123
21    472
22     24
23    104
24     28
25    101
26      2
27     31
Name: Count, dtype: int64


We separate the "yes" and "no" responses to the "Towel.Reuse" variable in the control group and save them as a separate array. We do the exact same for the social norm group.

In [3]:
# count has the number of yes and no for control and social norm groups
control_yes = count[::4].to_numpy() # every 4th starting from 0 - control group + yes
control_no = count[2::4].to_numpy() # every 4th starting from 2 - control group + no
control_total = np.array([y + n for y, n in zip(control_yes, control_no)])

print("The values of the control group:", control_total)

social_yes = count[1::4].to_numpy() # every 4th starting from 1 - social norm group + yes
social_no = count[3::4].to_numpy() # every 4th starting from 3 - social norm group + no
social_total = np.array([y + n for y, n in zip(social_yes, social_no)])

print("The values of the social norm group:", social_total)

The values of the control group: [211 277 135 187  25 147  30]
The values of the social norm group: [ 222 1318  655  555   24  576  132]


We add the variable "study", which refers to the number of the study the data belongs to. Additionally, we first combine the data into two separate data frames for the control and social norm groups. Lastly, we combine them into one. This allows us to keep the necessary information for our future model.

In [4]:
study = np.arange(1,len(control_yes)+1) # 7 diferent studies

control_data = pd.DataFrame({"reuse": control_yes, "total": control_total, "group": "control", "study": study})
social_data = pd.DataFrame({ "reuse": social_yes, "total": social_total, "group": "social", "study": study})

combined_data = pd.concat([control_data, social_data], ignore_index=True)

print("Data frame of the values to be used:")
print(combined_data)

Data frame of the values to be used:
    reuse  total    group  study
0      74    211  control      1
1     103    277  control      2
2      77    135  control      3
3      82    187  control      4
4      21     25  control      5
5     123    147  control      6
6      28     30  control      7
7      98    222   social      1
8     587   1318   social      2
9     406    655   social      3
10    278    555   social      4
11     21     24   social      5
12    472    576   social      6
13    101    132   social      7


The model will be defined using the function provided to us by the package "bambi". To this end, we transform the data frame into a version, which can be read by the package.

In [5]:
combined_data['reuse'] = combined_data['reuse'].astype(int)
combined_data['total'] = combined_data['total'].astype(int)
combined_data['group'] = combined_data['group'].astype('category')
combined_data['study'] = combined_data['study'].astype('category')

print("Altered data frame:")
print(combined_data)

Altered data frame:
    reuse  total    group study
0      74    211  control     1
1     103    277  control     2
2      77    135  control     3
3      82    187  control     4
4      21     25  control     5
5     123    147  control     6
6      28     30  control     7
7      98    222   social     1
8     587   1318   social     2
9     406    655   social     3
10    278    555   social     4
11     21     24   social     5
12    472    576   social     6
13    101    132   social     7


### Task 1

Based on the data, the binomial model is chosen for the model. For each observation (each study) $i$, let the number of towels used (reuse) be denoted by $y_i$, the number of guests (total) be $n_i$ and the probability of a guest reusing their towel be $p_i$. We can view this problem as a $i$ times repeated experiment of whether a guest reuses their towel (success) or not (failure).

\begin{equation*}
y_i \sim \text{Binomial}(n_i, p_i).
\end{equation*}

We take the logit link function to trnsform the probability $p_i\in (0, 1)$ into log-odds $\eta_i(-\infty, \infty)$. 

\begin{equation*}
\text{logit}(p_i)=\left(\log{\frac{p_i}{1-p_i}}\right) = \eta_i.
\end{equation*}

We set up a Bayesian hierarchical logistic regression model

\begin{equation*}
y_{jk} \sim \text{Binomial}(n_{jk}, p_{jk}),
\end{equation*}
and

\begin{equation*}
\text{logit}(p_{jk}) = \beta_0 + \beta_{social}\times 1(\text{group}_{jk}=\text{social} + u_j).
\end{equation*}

Here, $n_{jk}$, $y_{jk}$ and $p_{jk}$ are as before, with the indices referring to $j$ study and $k$ group. $\beta_0$ is the fixed intercept, representing towel reuse in the control group. $\beta_{social}$ is the effect of social group. Lastly, the $1(\text{group}_{jk}=\text{social})$ is the indicator function that is $1$ if the observation is from the social norm group and $0$ if it is from the control group and $u_j$ is the random intercept.


In [6]:
model = bmb.Model("p(reuse, total) ~ group + (1|study)", data=combined_data, family="binomial")

### Task 2

Between study heterogeneity is accounted for by the variable $u_j$, which allows each of the $7$ studies to have their own baseline log-odds and shifts the model's intercept value depending on the specific hotel's baseline. In practice this is done by including (1|study) in the argument of the model.


### Task 3

We estimate the parameters of our Bayesian inference by using Markov Chain Monte Carlo (MCMC) sampling. To be exact, we run $4$ independent chains with $2 000$ draws each, which results in $8 000$ draws in total. We set to seed to $42$ (just some arbitrary number), for reproducibility's sake. 

In [7]:
results = model.fit(draws=2000, chains=4, random_seed=42)
print(results)

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (4 chains in 1 job)
NUTS: [Intercept, group, 1|study_sigma, 1|study_offset]


Output()

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 283 seconds.
There were 120 divergences after tuning. Increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


<xarray.DataTree>
Group: /
├── Group: /posterior
│       Dimensions:            (chain: 4, draw: 2000, group_dim: 1, study__factor_dim: 7)
│       Coordinates:
│         * chain              (chain) int64 32B 0 1 2 3
│         * draw               (draw) int64 16kB 0 1 2 3 4 ... 1995 1996 1997 1998 1999
│         * group_dim          (group_dim) <U6 24B 'social'
│         * study__factor_dim  (study__factor_dim) <U1 28B '1' '2' '3' '4' '5' '6' '7'
│       Data variables:
│           Intercept          (chain, draw) float64 64kB 0.8662 0.8895 ... 0.4254
│           group              (chain, draw, group_dim) float64 64kB 0.2202 ... 0.2218
│           1|study_sigma      (chain, draw) float64 64kB 0.931 1.112 ... 1.015 1.276
│           1|study            (chain, draw, study__factor_dim) float64 448kB -1.263 ...
│       Attributes:
│           created_at:                  2026-09-19T11:08:34.436700+00:00
│           creation_library:            ArviZ
│           creation_library_version: 

### Task 4

In this task we make a summary of the parameter values by the posterior mean and $90\%$ probability interval. We create the summary, then get the posterior, for which we calculate the mean and the $90\%$ probability interval. Most importantly, we take the exponential of both to transform them back.

In [ ]:
summary = az.summary(results, ci_prob=0.9)

g_samples = results.posterior["group"].sel(group_dim="social").values.flatten()

mean_logodds = np.mean(g_samples)

ci_logodds = np.percentile(g_samples, [5, 95])

mean_odds = np.exp(mean_logodds)
ci_odds = np.exp(ci_logodds)

print(f"Log-odds mean: {mean_logodds:.4f}, 90% CI: [{ci_logodds[0]:.4f}, {ci_logodds[1]:.4f}]")
print(f"Odds Ratio mean: {mean_odds:.4f}, 90% CI: [{ci_odds[0]:.4f}, {ci_odds[1]:.4f}]")

Log-odds mean: 0.2116, 90% CI: [0.0834, 0.3389]
Odds Ratio mean: 1.2357, 90% CI: [1.0869, 1.4034]


### Task 5

We propose a hypothesis for testing the significance of the $\beta_{\text{social}}$ parameter.

\begin{align*}
H_0 :& \beta_{\text{social}} = 0 \\
H_1 :& \beta_{\text{social}} \neq 0 .
\end{align*}

### Task 6

With Bayesian hypothesis testing, we will calculate the posterior probability of the parameters given the observed data,

\begin{equation*}
P\left( H_1 | \text{Data}\right) = P\left( \beta_{\text{social}}>0 | \text{Data}\right) .
\end{equation*}

In [10]:
p_greater = np.mean(g_samples > 0)
p_two_side = 2 * min(p_greater, 1 - p_greater)

print(f"P(beta_group > 0 | Data) = {p_greater:.4f}")
print(f"Two-sided probability = {p_two_side:.4f}")

P(beta_group > 0 | Data) = 0.9952
Two-sided probability = 0.0095


From this, we can see that $\beta_{\text{social}}$ has a $99,52\%$ influence on the model, which means that we reject the null hypothesis. 